# 🔬 SEM Image Analysis: Food Microstructure
### FBMFOR — Food Fraud Analysis | University of Reading

This notebook loads scanning electron microscopy (SEM) images from `data/sem/`,
groups them by sample code, displays a contact-sheet gallery, and computes basic
image statistics (mean intensity, contrast, coefficient of variation) as simple
texture proxies.

SEM images are 16-bit greyscale TIFF files. The `.ipj` files in the same folder
are ImageJ project files used during acquisition; they are not processed here.


## 1 · Setup

In [ ]:
# ============================================================
# SETUP
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

try:
    import google.colab      # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    DATA_DIR   = Path('/content/data/sem')
    OUTPUT_DIR = Path('/content/data/processed')
    print("\u25b6 Running on Google Colab")
else:
    DATA_DIR   = Path('../data/sem')
    OUTPUT_DIR = Path('../data/processed')
    print("\u25b6 Running locally")

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Download TIF images from GitHub when running in Colab ─────
GITHUB_RAW = "https://raw.githubusercontent.com/ggkuhnle/fbmfor-foodfraud/main"

# All sample groups and their image indices
SEM_FILES = [
    f"{prefix}_{idx:03d}.tif"
    for prefix in ["CMA", "CMB", "CoA", "CoB", "GM", "GMS", "NB", "sand"]
    for idx in range(1, 5)
]

if IN_COLAB:
    import urllib.request
    missing = [f for f in SEM_FILES if not (DATA_DIR / f).exists()]
    if missing:
        print(f"  Downloading {len(missing)} TIF files from GitHub ...")
        for fname in missing:
            url = f"{GITHUB_RAW}/data/sem/{fname}"
            urllib.request.urlretrieve(url, DATA_DIR / fname)
        print(f"  Done \u2713")
    else:
        print("  Files already present \u2713")

print(f"   Data directory   : {DATA_DIR.resolve()}")
print(f"   Output directory : {OUTPUT_DIR.resolve()}")
print("\nSetup complete \u2713")


## 2 · Discover images

Images are named `<SampleCode>_NNN.tif` (e.g. `CMA_001.tif`).
We extract the sample code from the filename prefix.


In [ ]:
# ============================================================
# DISCOVER AND GROUP IMAGES
# ============================================================

tif_files = sorted(DATA_DIR.glob('*.tif'))

if not tif_files:
    raise FileNotFoundError(f"No .tif files found in {DATA_DIR.resolve()}")

# Group by prefix (everything before the last underscore + digits)
import re
from collections import defaultdict

groups = defaultdict(list)
for f in tif_files:
    # Match pattern: <anything>_<digits>
    m = re.match(r'^(.+?)_?(\d+)$', f.stem)
    prefix = m.group(1).rstrip('_') if m else f.stem
    groups[prefix].append(f)

groups = dict(sorted(groups.items()))

print(f"Found {len(tif_files)} images in {len(groups)} sample groups:\n")
for prefix, files in groups.items():
    print(f"  {prefix:<12}  {len(files)} image(s)  "
          f"[{', '.join(f.name for f in files)}]")


## 3 · Gallery — all samples

Each row is one sample; each column is one of its images.
Images are displayed with a shared greyscale colour scale per sample
so brightness differences between fields of view are visible.


In [ ]:
# ============================================================
# CONTACT-SHEET GALLERY
# ============================================================

# ── Load all images into memory ───────────────────────────────────────────────
def load_tif(path: Path) -> np.ndarray:
    """Load a TIFF to a float32 array normalised to [0, 1]."""
    img = np.array(Image.open(path)).astype(np.float32)
    lo, hi = img.min(), img.max()
    if hi > lo:
        img = (img - lo) / (hi - lo)
    return img

images = {prefix: [load_tif(f) for f in files]
          for prefix, files in groups.items()}

# ── Layout ────────────────────────────────────────────────────────────────────
n_rows = len(groups)
n_cols = max(len(v) for v in groups.values())

fig, axes = plt.subplots(n_rows, n_cols,
                         figsize=(3.5 * n_cols, 3.0 * n_rows),
                         squeeze=False)

for row_idx, (prefix, files) in enumerate(groups.items()):
    imgs = images[prefix]
    # Shared intensity range across images of the same sample
    vmin = min(im.min() for im in imgs)
    vmax = max(im.max() for im in imgs)

    for col_idx in range(n_cols):
        ax = axes[row_idx][col_idx]
        if col_idx < len(files):
            ax.imshow(imgs[col_idx], cmap='gray', vmin=vmin, vmax=vmax,
                      aspect='equal', interpolation='none')
            ax.set_title(files[col_idx].stem, fontsize=8, pad=3)
        else:
            ax.set_visible(False)

        ax.set_xticks([])
        ax.set_yticks([])

    # Row label on the left-most axis
    axes[row_idx][0].set_ylabel(prefix, fontsize=11, fontweight='bold',
                                rotation=90, labelpad=6)

fig.suptitle('SEM Gallery — Greyscale Contact Sheet', fontsize=14, y=1.01)
plt.tight_layout()
out_path = OUTPUT_DIR / 'sem_gallery.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved → {out_path}")
